# Salary EDA: факторы, связанные с уровнем зарплаты

Портфолио-версия exploratory data analysis учебного датасета о зарплатах.  
Цель: проверить качество данных, изучить распределения и взаимосвязи, а также описать верхний зарплатный сегмент и должности, где высокие зарплаты встречаются чаще внутри выборки.


## Исследовательские вопросы

1. Как распределены зарплаты и есть ли аномальные значения?
2. Как зарплата связана с возрастом и профессиональным опытом?
3. Как различается зарплата между уровнями образования?
4. Сохраняются ли различия между мужчинами и женщинами после учёта возраста, опыта и образования?
5. Чем верхние ~10% зарплат отличаются от остальных наблюдений?
6. В каких должностях высокие зарплаты встречаются чаще?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

df = pd.read_csv('data/salary_dataset.csv')
df.head()


## 1. Очистка данных

In [ ]:
# Удаляем технический индекс, который маскирует полные дубликаты
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns='Unnamed: 0')

df.columns = (
    df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
)

duplicates_before = df.duplicated().sum()
rows_before = len(df)

df = df.drop_duplicates().dropna().reset_index(drop=True)

df['education_level'] = df['education_level'].replace({
    "Bachelor's": "Bachelor's Degree",
    "Master's": "Master's Degree",
    "phD": "PhD"
})

# После объединения синонимичных категорий снова удаляем полные совпадения
df = df.drop_duplicates().reset_index(drop=True)

print(f'Исходных строк: {rows_before}')
print(f'Полных дубликатов после удаления технического индекса: {duplicates_before}')
print(f'После базовой очистки: {len(df)}')


In [ ]:
# Проверяем нижний хвост зарплаты
display(df.nsmallest(10, 'salary')[
    ['age', 'job_title', 'years_of_experience', 'salary', 'country']
])

# В исходном исследовании значения < 20 000 признаны аномальными
df_clean = df[df['salary'] >= 20_000].reset_index(drop=True)

print(f'Финальная аналитическая выборка: {len(df_clean)} наблюдений')


Ключевой результат очистки: после удаления технического индексного столбца обнаружены **1535 полных дубликатов**. Финальная аналитическая выборка содержит **5158 наблюдений**.


## 2. Распределение зарплаты и основные связи

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean['salary'], bins=35, kde=True, ax=axes[0])
axes[0].axvline(df_clean['salary'].median(), linestyle='--', label='Median')
axes[0].set_title('Распределение зарплаты')
axes[0].legend()

sns.scatterplot(
    data=df_clean,
    x='years_of_experience',
    y='salary',
    alpha=0.25,
    ax=axes[1]
)
axes[1].set_title('Зарплата и профессиональный опыт')

plt.tight_layout()
plt.show()

corr = df_clean[['age', 'years_of_experience', 'salary']].corr()
display(corr)


В очищенной выборке зарплата сильнее связана с профессиональным опытом (`r ≈ 0.814`), чем с возрастом (`r ≈ 0.743`).


## 3. Образование и опыт

In [ ]:
salary_by_education = (
    df_clean.groupby('education_level', observed=True)['salary']
    .agg(['count', 'mean', 'median'])
    .sort_values('median')
)
display(salary_by_education)

plt.figure(figsize=(9, 5))
sns.boxplot(data=df_clean, x='education_level', y='salary')
plt.xticks(rotation=20)
plt.title('Зарплата по уровню образования')
plt.tight_layout()
plt.show()


In [ ]:
model_experience = smf.ols(
    'salary ~ years_of_experience',
    data=df_clean
).fit()

model_education = smf.ols(
    'salary ~ C(education_level)',
    data=df_clean
).fit()

model_combined = smf.ols(
    'salary ~ years_of_experience + C(education_level)',
    data=df_clean
).fit()

pd.DataFrame({
    'Модель': ['Опыт', 'Образование', 'Опыт + образование'],
    'R²': [
        model_experience.rsquared,
        model_education.rsquared,
        model_combined.rsquared
    ]
})


По результатам OLS, модель с профессиональным опытом объясняет около **66.3%** вариации зарплаты, модель только с образованием около **41.9%**, а совместная модель около **71.6%**. Это описательная ассоциация, а не причинный эффект.


## 4. Гендерные различия

In [ ]:
df_gender = df_clean[df_clean['gender'].isin(['Male', 'Female'])].copy()

display(
    df_gender.groupby('gender')['salary']
    .agg(['count', 'mean', 'median'])
)

model_gender = smf.ols(
    'salary ~ C(gender) + years_of_experience + age + C(education_level)',
    data=df_gender
).fit()

model_gender.summary().tables[1]


После статистического контроля возраста, опыта и образования мужчины в выборке имеют в среднем примерно на **6.9 тыс.** более высокую зарплату. Результат описывает ассоциацию внутри датасета и не доказывает причинного влияния пола.


## 5. Верхний зарплатный дециль

In [ ]:
salary_threshold = df_clean['salary'].quantile(0.9)
df_clean['is_high_salary'] = (df_clean['salary'] >= salary_threshold).astype(int)

education_mapping = {
    'High School': 0,
    "Bachelor's Degree": 1/3,
    "Master's Degree": 2/3,
    'PhD': 1
}
df_clean['education_score'] = (
    df_clean['education_level'].map(education_mapping).astype(float)
)

salary_comparison = (
    df_clean.groupby('is_high_salary')
    .agg(
        age=('age', 'mean'),
        years_of_experience=('years_of_experience', 'mean'),
        education_score=('education_score', 'mean')
    )
    .T
)

salary_comparison.columns = ['Остальные ~90%', 'Топ ~10%']
salary_comparison['Разница'] = (
    salary_comparison['Топ ~10%'] - salary_comparison['Остальные ~90%']
)
display(salary_comparison)

print(f'Порог верхнего дециля: {salary_threshold:.0f}')


Верхний зарплатный сегмент в среднем старше, имеет примерно вдвое больший профессиональный опыт и более высокий уровень образования.


## 6. Должности и высокий уровень зарплаты

In [ ]:
job_salary_profile = (
    df_clean.groupby('job_title')
    .agg(
        employees=('salary', 'size'),
        median_salary=('salary', 'median'),
        high_salary_count=('is_high_salary', 'sum')
    )
)

job_salary_profile['high_salary_share_%'] = (
    job_salary_profile['high_salary_count']
    / job_salary_profile['employees']
    * 100
).round(1)

jobs = (
    job_salary_profile[job_salary_profile['employees'] >= 10]
    .sort_values(
        ['high_salary_share_%', 'high_salary_count'],
        ascending=False
    )
)

display(jobs.head(15))


Среди должностей минимум с 10 наблюдениями особенно выделяются `Director of Data Science`, `Marketing Director` и `Software Engineer Manager` по доле работников в верхнем зарплатном дециле. По абсолютному числу наблюдений из верхнего дециля лидируют `Software Engineer Manager`, `Senior Project Engineer`, `Data Scientist` и `Product Manager`.

Это описание фактически наблюдаемого распределения в выборке, а не прогноз зарплаты для нового человека.


## Итог

- качество исходных данных существенно зависело от корректного удаления технического индекса;
- профессиональный опыт оказался наиболее сильным из рассмотренных числовых факторов, связанных с зарплатой;
- образование добавляет самостоятельную информацию после учёта опыта;
- верхний зарплатный сегмент отличается прежде всего опытом, возрастом, образованием и профессиональной специализацией;
- выводы относятся только к данной выборке: неизвестны валюта, период начисления зарплаты и способ формирования датасета, поэтому результаты нельзя напрямую переносить на рынок труда.
